# Day 2 — Function calling: the model asks, your code runs (Gemini)
**Theory:** [day-02-tool-calling.md](../theory/day-02-tool-calling.md) · ⏱ ~60 min · Level: easy → medium

**You will:**
1. Do one function call fully by hand and inspect every part
2. Handle **parallel** function calls and **errors**
3. Use **structured output** (Pydantic schema → guaranteed JSON)
4. Run a tool-design A/B test (bad tool vs good tool)

> The google-genai SDK can also call Python functions for you ("automatic function calling", Day 14).
> Here we do it **by hand** so you see the protocol.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. A real tool over the ShopEase database
The function is plain Python. The *declaration* (name, description, JSON schema) is all the model sees.

In [ ]:
import json, sqlite3
from course_utils import function_response
DB = "projects/shopease-support-agent/data/shopease.db"

def get_order(order_id: str) -> str:
    row = sqlite3.connect(DB).execute(
        "SELECT id, item, amount, status, delivered_on FROM orders WHERE id=?", (order_id,)).fetchone()
    if not row:
        return f"Error: no order {order_id}. Order ids look like 'O2001'."
    return json.dumps(dict(zip(["id", "item", "amount", "status", "delivered_on"], row)))

GET_ORDER = types.FunctionDeclaration(
    name="get_order",
    description="Look up one ShopEase order by id (e.g. 'O2001'). Returns item, amount, status and delivery date.",
    parameters_json_schema={"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
)
CONFIG = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[GET_ORDER])], thinking_config=thinking("low"))
print(get_order("O2001"))

## 2. One round trip, by hand

In [ ]:
contents = [user("What's the status of order O2001?")]
r = client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
tracker.add(MODEL, r.usage_metadata, "call 1")
for p in r.candidates[0].content.parts:
    kind = "function_call" if p.function_call else "thought" if p.thought else "text"
    print(" ", kind, "→", p.function_call or (p.text or "")[:80])

In [ ]:
call = r.function_calls[0]
result = get_order(**call.args)                       # YOUR code runs the tool
contents.append(r.candidates[0].content)              # the model turn, exactly as received (keeps thought signatures)
contents.append(types.Content(role="user", parts=[function_response(call, result)]))  # matched by name + id
final = client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
tracker.add(MODEL, final.usage_metadata, "call 2")
print(text_of(final))

## 3. Parallel calls + errors as feedback
Ask about three orders, one of which doesn't exist. Run all calls, return **all results in ONE turn**,
and send failures as `{"error": ...}` so the model can recover.

In [ ]:
def run_tools(response):
    parts = []
    for fc in response.function_calls or []:
        out = get_order(**fc.args)
        print(f"   🔧 {fc.name}({dict(fc.args)}) → {out[:60]}")
        parts.append(function_response(fc, out, is_error=out.startswith("Error")))
    return parts

contents = [user("Compare orders O2001, O2004 and O9999 in one short table.")]
for step in range(5):
    r = client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
    tracker.add(MODEL, r.usage_metadata, f"parallel step {step}")
    contents.append(r.candidates[0].content)
    print(f"step {step}: {len(r.function_calls or [])} function call(s) in one reply")
    if not r.function_calls:
        break
    contents.append(types.Content(role="user", parts=run_tools(r)))
print("\n" + text_of(r))

## 4. Structured output: guaranteed-valid JSON
Pass a Pydantic model as `response_schema` and read `response.parsed`. Perfect for classification and extraction.

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel

class Ticket(BaseModel):
    category: Literal["order_status", "refund", "warranty", "other"]
    order_id: Optional[str]
    urgency: Literal["low", "medium", "high"]

tickets = ["My watch from order O2001 stopped working!! need help asap",
           "where's my package O2003, it's been weeks",
           "do you sell gift cards?"]
for t in tickets:
    r = client.models.generate_content(model=CHEAP_MODEL, contents=t, config=types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=Ticket))
    tracker.add(CHEAP_MODEL, r.usage_metadata, "structured")
    print(f"{r.parsed!r:<75} ← {t}")

## 5. Tool design A/B: "dump everything" vs "task-shaped search"
Same question, two tool designs. Compare tokens and correctness. (`run_agent` is the loop above, packaged.)

In [ ]:
from course_utils import run_agent

def list_all_orders() -> str:
    return json.dumps(sqlite3.connect(DB).execute("SELECT * FROM orders").fetchall())

def search_orders(customer_id: str, status: str = "") -> str:
    q, args = "SELECT id, item, status FROM orders WHERE customer_id=?", [customer_id]
    if status:
        q, args = q + " AND status=?", args + [status]
    return json.dumps(sqlite3.connect(DB).execute(q, args).fetchall())

TOOL_A = [{"name": "list_all_orders", "description": "Return every order in the database.",
           "input_schema": {"type": "object", "properties": {}}}]
TOOL_B = [{"name": "search_orders", "description": "Search one customer's orders, optionally by status "
           "(processing/shipped/delivered/returned).",
           "input_schema": {"type": "object", "properties": {"customer_id": {"type": "string"}, "status": {"type": "string"}},
                            "required": ["customer_id"]}}]
question = "Which orders of customer C002 are still shipped (in transit)?"
for name, tools, fns in [("A: dump all", TOOL_A, {"list_all_orders": list_all_orders}),
                         ("B: search", TOOL_B, {"search_orders": search_orders})]:
    before = len(tracker.calls)
    res = run_agent(question, tools, fns, model=MODEL, thinking_level="low", verbose=False)
    used = sum(c["input"] for c in tracker.calls[before:])
    print(f"{name:<12} input tokens={used:>6,}  steps={res.steps}  answer: {res.text[:90]!r}")

## ✏️ Exercises
1. In section 3, send errors as `{"output": ...}` instead of `{"error": ...}`. Does the model still recover from O9999?
2. Make `get_order`'s error message unhelpful (`"Error 404"`). Does the model behave worse?
3. Add a `check_refund_eligibility` tool that encodes the 30-day rule in code. Why is code better than a prompt rule here?

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None